# Notebook 1: Data Validation and Preprocessing

## Purpose
Ensure clean, normalized data, build derived indices, segment books.

## Tasks
1. Validate and load data files
2. Normalize topic probabilities (per book/segment sum to 1)
3. Merge topic proportions with metadata
4. Segment books into begin/middle/end if chapter-level topic probs missing
5. Compute derived composite category proportions per book

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 1. Define Paths and Load Configuration

In [ ]:
# Define project root and data paths
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent

# Input paths
BOOK_CAT_PROPS = PROJECT_ROOT / "results" / "stage09_category_mapping" / "stage2_theory_driven_categories" / "book_category_proportions.parquet"
TAXONOMY_MAPPINGS = PROJECT_ROOT / "results" / "stage09_category_mapping" / "stage2_theory_driven_categories" / "taxonomy_mappings_*.json"
RADWAY_MAPPINGS = PROJECT_ROOT / "results" / "stage09_category_mapping" / "stage3_radway_functions" / "taxonomy_with_radway.json"
GOODREADS_CSV = PROJECT_ROOT / "data" / "processed" / "goodreads.csv"
CHAPTERS_CSV = PROJECT_ROOT / "data" / "processed" / "chapters.csv"

# Output paths
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Load and Validate Data Files

In [ ]:
# Load book category proportions
if BOOK_CAT_PROPS.exists():
    book_cat_props = pd.read_parquet(BOOK_CAT_PROPS)
    print(f"✓ Loaded book category proportions: {len(book_cat_props)} rows")
    print(f"  Columns: {list(book_cat_props.columns)}")
    print(f"  Books: {book_cat_props['book_id'].nunique()}")
else:
    print(f"⚠ Warning: {BOOK_CAT_PROPS} not found")
    book_cat_props = None

In [ ]:
# Load taxonomy mappings
import glob
taxonomy_files = list(glob.glob(str(TAXONOMY_MAPPINGS)))
if taxonomy_files:
    taxonomy_mapping = json.load(open(taxonomy_files[0], 'r'))
    print(f"✓ Loaded taxonomy mappings: {len(taxonomy_mapping)} topics")
else:
    print(f"⚠ Warning: No taxonomy mapping files found")
    taxonomy_mapping = None

In [ ]:
# Load Radway mappings
if Path(RADWAY_MAPPINGS).exists():
    radway_mapping = json.load(open(RADWAY_MAPPINGS, 'r'))
    print(f"✓ Loaded Radway mappings")
else:
    print(f"⚠ Warning: {RADWAY_MAPPINGS} not found")
    radway_mapping = None

In [ ]:
# Load Goodreads metadata
if GOODREADS_CSV.exists():
    books_meta = pd.read_csv(GOODREADS_CSV)
    print(f"✓ Loaded Goodreads metadata: {len(books_meta)} books")
    print(f"  Columns: {list(books_meta.columns)}")
else:
    print(f"⚠ Warning: {GOODREADS_CSV} not found")
    books_meta = None

## 3. Normalize Topic Probabilities

In [ ]:
# TODO: Normalize probabilities per book/segment
# Ensure probabilities sum to 1.0 for each book

if book_cat_props is not None:
    # Check if proportions already sum to 1.0
    prop_sums = book_cat_props.groupby('book_id')['prop'].sum()
    print(f"Proportion sums - Min: {prop_sums.min():.4f}, Max: {prop_sums.max():.4f}, Mean: {prop_sums.mean():.4f}")
    
    # Normalize if needed
    if not np.allclose(prop_sums, 1.0, atol=0.01):
        print("Normalizing proportions...")
        book_cat_props['prop'] = book_cat_props.groupby('book_id')['prop'].transform(
            lambda x: x / x.sum()
        )

## 4. Merge with Metadata

In [ ]:
# TODO: Merge category proportions with book metadata
# Create popularity groups (Top/Mid/Trash) based on ratings

if book_cat_props is not None and books_meta is not None:
    # Merge
    merged_df = book_cat_props.merge(
        books_meta,
        on='book_id',
        how='inner'
    )
    print(f"✓ Merged dataset: {len(merged_df)} rows")
    
    # Create popularity groups if not already present
    if 'group' not in merged_df.columns and 'average_rating_weighted_mean' in merged_df.columns:
        # Define quartiles or terciles
        # TODO: Adjust based on your grouping strategy
        pass

## 5. Segment Books (if chapter-level data missing)

In [ ]:
# TODO: If chapter_topic_probs.csv is missing, segment books into begin/middle/end
# Split each book's token stream into tertiles and re-infer topic mixtures per tertile

# This would require:
# 1. Loading sentence-level data with topics
# 2. Grouping by book_id and ordering by position
# 3. Splitting into tertiles
# 4. Computing topic proportions per tertile

## 6. Compute Composite Category Proportions

In [ ]:
# TODO: Compute derived composite category proportions per book
# This will be used in subsequent notebooks for index computation

## 7. Save Outputs

In [ ]:
# Save cleaned datasets
if 'merged_df' in locals():
    output_file = OUTPUT_DIR / "book_category_props.csv"
    merged_df.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")

# TODO: Save chapter_category_props.csv if available

## Summary

Data validation and preprocessing complete. Next: Notebook 2 (Index Computation)